# CelebA-HQ Face Generation — DDPM

Generates human face images from a saved checkpoint using the trained DDPM model.

**Prerequisites:**
- A checkpoint file (`ddpm_faces_final.pth`) already saved to Google Drive by the training notebook.
- The repo cloned at the path below (done automatically by this notebook).

**To generate for a different run**, change only `RUN_NAME` and re-run all cells.


## 1 · Mount Drive and clone repo


In [ ]:
import os
import shutil
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

# ── Repo ────────────────────────────────────────────────────────────────────
REPO_URL = "https://github.com/AAnanya19/Human-Faces-Generation-Diffusion-Models.git"
BRANCH   = "main"
WORKDIR  = Path("/content/Human-Faces-Generation-Diffusion-Models")

# ── Run config ──────────────────────────────────────────────────────────────
# Change RUN_NAME to point at a different training run.
CHECKPOINTS_ROOT = Path("/content/drive/MyDrive/aml/ddpm_runs")
RUN_NAME         = "faces_run_001"          # ← edit this
CHECKPOINT_PATH  = CHECKPOINTS_ROOT / RUN_NAME / "ddpm_faces_final.pth"
OUTPUT_DIR       = CHECKPOINTS_ROOT / RUN_NAME / "generated_faces"

# ── Clone ───────────────────────────────────────────────────────────────────
if WORKDIR.exists():
    shutil.rmtree(WORKDIR)

!git clone --branch "$BRANCH" "$REPO_URL" "$WORKDIR"

os.chdir(WORKDIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(
        f"Checkpoint not found: {CHECKPOINT_PATH}\n"
        f"Make sure the training notebook has finished and saved to Drive."
    )

print("Project root  :", WORKDIR)
print("Checkpoint    :", CHECKPOINT_PATH)
print("Output dir    :", OUTPUT_DIR)


## 2 · Install dependencies


In [ ]:
assert os.path.isfile("requirements.txt"), "Run the clone cell first."
!pip install -q datasets huggingface_hub tqdm matplotlib torchvision


## 3 · Generation config

These values **must match** the hyperparameters used during training.

| Parameter | Default | Notes |
|---|---|---|
| `NUM_IMAGES` | 300 | Required for FID evaluation against the 300-image test set |
| `BATCH_SIZE` | 16 | Reduce to 8 if you hit GPU OOM |
| `IMAGE_SIZE` | 256 | Must match training |
| `TIMESTEPS` | 1000 | Must match training |
| `BASE_CHANNELS` | 128 | Must match U-Net training config |
| `TIME_DIM` | 512 | Must match U-Net training config |


In [ ]:
NUM_IMAGES    = 300   # total images to generate (matches FID test-set size)
BATCH_SIZE    = 16    # images per forward pass — reduce to 8 if OOM
IMAGE_SIZE    = 256   # must match training
TIMESTEPS     = 1000  # must match training
BASE_CHANNELS = 128   # U-Net base channel width — must match training
TIME_DIM      = 512   # time-embedding dimension — must match training


## 4 · Verify GPU


In [ ]:
import torch

print("torch  :", torch.__version__)
print("device :", "cuda" if torch.cuda.is_available() else "cpu")

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is not active.\n"
        "Go to Runtime → Change runtime type → T4 GPU (or A100 if available)."
    )

# Show VRAM — useful to decide whether to reduce BATCH_SIZE
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader


## 5 · Generate images

Individual PNGs are saved to `OUTPUT_DIR` for FID evaluation.  
A preview grid (`faces_grid.png`) is also saved for quick inspection.


In [ ]:
!python3 scripts/generate_faces.py \
    --checkpoint  "$CHECKPOINT_PATH"  \
    --out_dir     "$OUTPUT_DIR"       \
    --num_images  $NUM_IMAGES         \
    --batch_size  $BATCH_SIZE         \
    --image_size  $IMAGE_SIZE         \
    --timesteps   $TIMESTEPS          \
    --base_channels $BASE_CHANNELS    \
    --time_dim    $TIME_DIM           \
    --device      cuda


## 6 · Preview the generated grid


In [ ]:
from IPython.display import Image, display

grid_path = OUTPUT_DIR / "faces_grid.png"
if not grid_path.is_file():
    raise FileNotFoundError(
        f"Preview grid not found: {grid_path}\n"
        f"Check the generation cell above for errors."
    )

print(f"Displaying preview grid from: {grid_path}")
display(Image(filename=str(grid_path)))


## 7 · Quick per-image browse (optional)

Flick through the first `N_PREVIEW` individual images inline — useful for spotting artefacts before running FID.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

N_PREVIEW = 16  # how many individual images to display here

png_files = sorted(OUTPUT_DIR.glob("face_*.png"))[:N_PREVIEW]
if not png_files:
    print("No generated images found — run the generation cell first.")
else:
    cols = 4
    rows = (len(png_files) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = axes.flatten()
    for ax, fpath in zip(axes, png_files):
        ax.imshow(mpimg.imread(fpath))
        ax.axis("off")
        ax.set_title(fpath.stem, fontsize=7)
    for ax in axes[len(png_files):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    print(f"Showing {len(png_files)} of {len(sorted(OUTPUT_DIR.glob('face_*.png')))} total images.")


---
**Next step:** run `scripts/evaluate_fid.py` pointing at `OUTPUT_DIR` and your test-set folder to get the FID score.

To generate for a different run or hyperparameter sweep, change `RUN_NAME` (cell 1) and/or the config values (cell 3) and re-run all cells.
